# MoE load-balancing lab

Why do Mixture-of-Experts routers pile tokens onto a few experts, and which fixes produce a model that actually
uses all of its experts? This notebook runs the experiments in `experiments/` (see `README.md` for the full
write-up and results).

1. **E1 – mechanisms**: causal interventions on the router ↔ expert feedback loop (synthetic task with known ground truth).
2. **E2 – fixes**: aux loss, z-loss, noise, aux-loss-free bias, expert choice, Sinkhorn, cosine router, dead-expert revival, hash routing.
3. **E3 – language model**: the same questions in a tiny byte-level MoE transformer on 3 text domains (use a GPU runtime).

In [ ]:
# Clone the repo. For a private repo, use a token: https://<TOKEN>@github.com/westhauxstripclub/colab.git
!git clone -b claude/moe-load-balancing-overfitting-grpouv https://github.com/westhauxstripclub/colab.git moe-lab
%cd moe-lab
!pip -q install -r requirements.txt
!python -m pytest -q tests

## Quick look: one run without balancing vs. one with the Switch aux loss

In [ ]:
import sys; sys.path.insert(0, '.')
import matplotlib.pyplot as plt
from moe_lab.train_synthetic import SynthRunConfig, run_synthetic

base = dict(moe=dict(d_hidden=8, n_experts=32), task=dict(n_clusters=32, n_train=32000, n_val=8000), seed=0)
runs = {name: run_synthetic(SynthRunConfig(name=name, **{**base, 'moe': {**base['moe'], **extra}}))
        for name, extra in [('none', {}), ('aux_1e-1', dict(aux_loss_coef=0.1))]}

fig, axes = plt.subplots(1, 2, figsize=(12, 3.5), sharey=True)
for ax, (name, r) in zip(axes, runs.items()):
    f = r['final']
    ax.bar(range(len(f['load'])), sorted(f['load'], reverse=True))
    ax.set_title(f"{name}: val loss {f['val_loss']:.3f}, dead experts {f['dead_experts']}, "
                 f"task info {f['label_info_recovered']:.2f}")
    ax.set_xlabel('expert (sorted by load)'); ax.set_ylabel('share of tokens')
plt.show()

## Full experiments
Results are cached per run under `results/`, so cells can be re-run to resume. Colab CPUs are slow: use fewer
seeds (`--seeds 4`) for a quick pass. E3 runs on the GPU.

In [ ]:
%cd experiments
!python e1_mechanisms.py --seeds 8 --workers 2
!python e2_fixes.py --seeds 8 --workers 2

In [ ]:
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
!python e3_lm.py --seeds 2 --steps 2500 --workers 1 --threads 2 --device {device}

In [ ]:
!python analyze.py
from IPython.display import Image, Markdown, display
for name in ['summary_e1.md', 'summary_e2.md', 'summary_e3.md']:
    display(Markdown(open(f'../results/{name}').read()))
import glob
for p in sorted(glob.glob('../results/figures/*.png')):
    display(Image(p))